# GSSC-S2D2 Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BillyChern/GSSC-S2D2/blob/main/examples/quickstart.ipynb)

Five-minute tour of the public API: load the headline checkpoint, sample one frame, visualise the result. Runs on a single GPU (any CUDA 12.x card with >= 12 GB) or CPU (slow).

**What you'll do:**
1. Install GSSC-S2D2 and its `spconv-cu126` companion (~ 2 min).
2. Load the released `gssc_mf/gssc_31k_mf_step40000/model_ema.safetensors` deployment checkpoint.
3. Run a single S2D2 correction step on a SemanticKITTI val frame.
4. Compare base SCPNet vs. S2D2 prediction vs. ground truth on a rare class.

**What you'll need on disk:** the headline checkpoint, ONE SCPNet base-prediction frame, and the matching raw SemanticKITTI voxel frame (`.bin` + `.label`). The base prediction is a narrowed fetch -- `scripts/download_assets.py` takes `--include`, so you do not need the whole 177 GiB / 190 GB prefix:

```
python scripts/download_assets.py --checkpoints
python scripts/download_assets.py --predictions --include 'scpnet_predictions/08/000000_*'
```

The raw SemanticKITTI voxels are NOT hosted with the release assets: they require registration at semantic-kitti.org, and `docs/DATASET.md` has the layout. Set `DATA_ROOT` below if yours lives elsewhere.

**What you won't need:**
- The full synthetic pool -- inference only.
- Multi-GPU -- single GPU is enough.

> Checkpoints, base-model predictions and the object bank are provisioned by `scripts/download_assets.py`; the synthetic pool's IEEE DataPort DOI has not been minted yet, so `scripts/download_assets.py --synthetic-pool` fails loudly with a `docs/DATASET.md` pointer instead of downloading. Nothing is withheld pending paper acceptance. See `docs/DATASET.md` for sizes and layout.


## 1. Install

Skip the cell below if running locally and you've already done `uv sync + uv pip install spconv-cu126==2.3.8`.

In [ ]:
import os
import subprocess
import sys

if 'COLAB_GPU' in os.environ:
    if not os.path.exists('GSSC-S2D2'):
        subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/BillyChern/GSSC-S2D2.git'])
    os.chdir('GSSC-S2D2')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'spconv-cu126==2.3.8'])

sys.path.insert(0, 'src')
import gssc

print(f'GSSC-S2D2 v{gssc.__version__} loaded')

## 2. Resolve assets

`scripts/download_assets.py` provisions the checkpoint and the base predictions, and when an asset is still missing the cell below will print a `docs/DATASET.md` pointer. The one asset the script cannot fetch is the synthetic pool, whose IEEE DataPort DOI is not minted yet -- inference does not need it. If you already have local assets, set `DATA_ROOT` to your data directory.

In [ ]:
from pathlib import Path

DATA_ROOT = Path(os.environ.get('DATA_ROOT', 'data'))
# v1.1.0 nested HF layout: data/checkpoints/<group>/<name>/model_ema.safetensors
# (matches scripts/download_assets.py output and docs/MODEL_ZOO.md)
CKPT      = DATA_ROOT / 'checkpoints' / 'gssc_mf' / 'gssc_31k_mf_step40000' / 'model_ema.safetensors'
FRAME_ID  = '000000'

voxel_path = DATA_ROOT / 'SemanticKITTI' / 'sequences' / '08' / 'voxels' / f'{FRAME_ID}.bin'
scp_path   = DATA_ROOT / 'scpnet_predictions' / '08' / f'{FRAME_ID}_pred.npy'
gt_path    = DATA_ROOT / 'SemanticKITTI' / 'sequences' / '08' / 'voxels' / f'{FRAME_ID}.label'

for p in [CKPT, voxel_path, scp_path, gt_path]:
    if not p.exists():
        raise FileNotFoundError(
            f'Missing asset: {p}. See docs/DATASET.md for download instructions '
            f'(scripts/download_assets.py provisions the checkpoints and base predictions).'
        )
print('All assets resolved')

## 3. Build the model + S2D2 correction sampler

In [ ]:
import numpy as np
import torch
from safetensors.torch import load_file

from gssc.diffusion.multinomial import MultinomialDiffusion3DV2
from gssc.inference.generate_predictions import load_lidar_voxels
from gssc.models.s2d2_unet import SceneCompletionUNetSparse

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# NOTE: "Sparse" in SceneCompletionUNetSparse refers to the auxiliary LiDAR
# encoder (spconv), NOT the denoiser. The released denoiser is a DENSE 3D U-Net
# (Conv3d) with additive/AdaGN conditioning (~35M reproduction).
model = SceneCompletionUNetSparse(
    num_classes=20, base_channels=32, time_emb_dim=128,
    lidar_base_channels=16, lidar_out_channels=32, lidar_in_channels=1,
    no_bev=False, ssc_cond_channels=20, ssc_multiscale=False,
).to(device)

# model_ema.safetensors already holds the EMA deployment weights (paper convention).
state = load_file(str(CKPT))
# strict=False tolerates buffer-only differences, but its result must be READ: the
# architecture three lines above is hardcoded, so a future checkpoint or a changed
# module layout would otherwise leave tensors at random init and still print a
# plausible IoU. Assert instead of discarding the _IncompatibleKeys.
load_result = model.load_state_dict(state, strict=False)
assert not load_result.missing_keys and not load_result.unexpected_keys, (
    f'checkpoint does not match this architecture: '
    f'{len(load_result.missing_keys)} missing, '
    f'{len(load_result.unexpected_keys)} unexpected -> {load_result}'
)
model.train(False)
diffusion = MultinomialDiffusion3DV2(num_classes=20, num_timesteps=100, beta_max=0.1).to(device)
print(f'Loaded EMA deployment weights from {CKPT.name} ({len(state)} tensors)')

## 4. Run a single S2D2 correction step

In [ ]:
import torch.nn.functional as F

lidar = load_lidar_voxels(str(voxel_path)).to(device)
scp   = np.load(scp_path).astype(np.int64)
scp_t = torch.from_numpy(scp).unsqueeze(0).to(device)
scp_oh = F.one_hot(scp_t, 20).float().permute(0, 4, 1, 2, 3)

bev = torch.zeros(1, 256, 256, dtype=torch.long, device=device)
for z in range(scp_t.shape[3] - 1, -1, -1):
    layer = scp_t[0, :, :, z]
    mask = (layer > 0) & (bev[0] == 0)
    bev[0][mask] = layer[mask]

with torch.no_grad():
    pred = diffusion.sample_algo2(
        model, bev, lidar, scpnet_pred=scp_t,
        shape=(1, 256, 256, 32), device=device,
        n_steps=1, show_progress=False, ssc_pred=scp_oh,
    )
pred_train = pred.cpu().numpy()[0]
print(f'Pred range: {pred_train.min()}-{pred_train.max()}')
print(f'Occupancy: {(pred_train > 0).sum() / pred_train.size:.2%}')

## 5. Compare against SCPNet base + ground truth

In [ ]:
from gssc.inference.generate_predictions import LEARNING_MAP_INV

lut = np.zeros(256, dtype=np.int64)
for ti, raw in enumerate(LEARNING_MAP_INV):
    lut[int(raw)] = ti
gt_raw   = np.fromfile(gt_path, dtype=np.uint16).reshape(256, 256, 32)
gt_train = lut[np.clip(gt_raw, 0, 255).astype(np.int64)]

def iou(pred: np.ndarray, gt: np.ndarray, cls: int) -> float:
    inter = ((pred == cls) & (gt == cls)).sum()
    union = ((pred == cls) | (gt == cls)).sum()
    return float(inter / union) if union > 0 else float('nan')

for cls, name in [(8, 'motorcyclist'), (18, 'pole'), (1, 'car')]:
    base = iou(scp, gt_train, cls)
    ours = iou(pred_train, gt_train, cls)
    print(f'  {name:<14s}  SCPNet: {base*100:5.2f}%  S2D2: {ours*100:5.2f}%  Delta {(ours-base)*100:+.2f}')

## What's next

- Full 4 071-frame eval: `python scripts/eval.py eval/val_1step --checkpoint data/checkpoints/gssc_mf/gssc_31k_mf_step40000/model_ema.safetensors` (~ 6 min on H100, expected **38.54 % mIoU**)
- D4 TTA: `python scripts/eval.py eval/val_d4tta --checkpoint data/checkpoints/gssc_mf/gssc_31k_mf_step40000/model_ema.safetensors` (*N* = 4 + D4, the chip configuration of section 6; ~ 3 h, expected **38.73 % mIoU**)
- BEV second task: `python scripts/eval.py eval/bev_secondary --checkpoint data/checkpoints/bev/bev_s2d2_scpnet/model.safetensors` -- paper `tab:bev_results` (supplementary Tab. XXI) -- expected **36.1 % BEV mIoU** (34.8 % parameter-free projection + 1.3). Use that checkpoint, not `bev/bev_perception_net` -- that is a different 938K-param 2D refinement model the BEV evaluator cannot load (see `docs/MODEL_ZOO.md`). The number is scored by the training-time 2D BEV evaluator on 100 fixed val frames (seed 42), *not* the 4,071-frame `semantic-kitti-api` protocol the rows above use, so it is not comparable with them
- Headline retrain: `python scripts/train.py train/31k_mf` (~ 37 GPU-hours on 2x H100; see [`docs/REPRODUCIBILITY.md`](../docs/REPRODUCIBILITY.md) for the single source of truth)

See [`docs/REPRODUCIBILITY.md`](../docs/REPRODUCIBILITY.md) for the full reproduction matrix.

## 6. Reproduce the paper's per-scene rare-class chips

The paper's reproduction matrix (supplementary Tab. XXIX, `tab:supp_repro_matrix`) routes
one row to this notebook, and it labels that row **N = 4 correction steps + D4 TTA** -- not
the single step of section 4. The raw voxel counts behind the two chips are supplementary
Tab. XIV (`tab:supp_fp_counts`), whose own note, and the caption of main-paper Fig. 6, state
the same configuration:

| class | frame | TP | FP | GT | IoU |
|---|---|---|---|---|---|
| bicyclist | `003096` | 2,724 | 1,956 | 2,833 | **56.9 %** |
| motorcyclist | `001417` | 255 | 94 | 315 | **62.3 %** |

`IoU = TP / (TP + FP + FN)` with `FN = GT - TP`, over the single class in the single frame.

So this section *does* need an extra run: section 4's prediction is the *N* = 1 headline
operating point and will not reproduce these chips. The first cell below re-samples the
current frame at `N = 4` with the full 8-element D4 group -- the same pair
`configs/eval/val_d4tta.yaml` and `configs/infer/val_d4tta.yaml` declare
(`correction_steps: 4`, `tta: d4`) -- and the second scores the chip. That is 32 forward
passes, roughly 32x the cost of section 4.

Point `FRAME_ID` (section 2) at one of the two ids, re-run sections 2-5 for that frame, then
run the two cells below. The scoring below covers the whole 256x256x32 grid and applies no
`.invalid` mask, so read it as a check on the configuration rather than a bit-exact
reproduction of the paper's counts.

In [ ]:
# The paper's chip configuration: N = 4 correction steps + the 8-element D4 group.
# supplementary Tab. XXIX (`tab:supp_repro_matrix`) labels the chip row that way, and
# configs/{eval,infer}/val_d4tta.yaml declare the same pair (correction_steps: 4,
# tta: d4). The transforms, their inverses and the per-transform BEV derivation below
# are the release's own D4 path -- the code `scripts/infer.py infer/val_d4tta` runs --
# not a notebook-local reimplementation.
from gssc.inference.d4_tta import (
    D4_ELEMENTS, apply_d4, derive_bev, invert_d4, run_algo2_softmax,
)

CHIP_STEPS = 4

soft_sum = None
for flip_x, flip_y, rot_k in D4_ELEMENTS:
    # derive_bev runs AFTER the transform, so the BEV stays consistent with the
    # transformed base. The zeros tensor is only the placeholder apply_d4 needs.
    lidar_t, base_t, _ = apply_d4(
        lidar, scp_t, torch.zeros(1, 256, 256, dtype=torch.long, device=device),
        flip_x, flip_y, rot_k,
    )
    soft = run_algo2_softmax(
        model, diffusion, lidar_t, base_t, derive_bev(base_t), CHIP_STEPS, device,
    )
    soft_back = invert_d4(soft, flip_x, flip_y, rot_k)
    soft_sum = soft_back if soft_sum is None else soft_sum + soft_back

pred_chip = (soft_sum / len(D4_ELEMENTS)).argmax(dim=1).cpu().numpy()[0]
print(f'chip config: N={CHIP_STEPS} + D4 TTA over {len(D4_ELEMENTS)} transforms')
print(f'Occupancy: {(pred_chip > 0).sum() / pred_chip.size:.2%}')

In [ ]:
# Per-scene, per-class IoU for the rare-class chips of supplementary Tab. XIV (`tab:supp_fp_counts`).
# `pred_chip` is the N=4 + D4-TTA prediction from the cell above -- the configuration the
# paper's chips use; `gt_train` comes from section 5. Both are (256,256,32) learning-map ids.

PAPER = {  # class -> (frame, TP, FP, GT_total, IoU %)
    'bicyclist':    ('003096', 2724, 1956, 2833, 56.9),
    'motorcyclist': ('001417',  255,   94,  315, 62.3),
}
CLASS_ID = {'bicyclist': 7, 'motorcyclist': 8}  # learning-map ids


def per_class_iou(pred, gt, class_id):
    """TP/FP/FN and IoU for one class in one frame, over the whole grid (no `.invalid` mask)."""
    p, g = (pred == class_id), (gt == class_id)
    tp, fp, fn = int((p & g).sum()), int((p & ~g).sum()), int((~p & g).sum())
    denom = tp + fp + fn
    return tp, fp, fn, (100.0 * tp / denom if denom else float('nan'))


for name, (frame, tp_ref, fp_ref, gt_ref, iou_ref) in PAPER.items():
    if frame != FRAME_ID:
        print(f'{name:13s} skipped - set FRAME_ID = {frame!r} in section 2 and re-run sections 2-5 and both cells of section 6 to check it')
        continue
    tp, fp, fn, iou = per_class_iou(pred_chip, gt_train, CLASS_ID[name])
    print(f'{name:13s} frame {frame}  (N=4 + D4 TTA)')
    print(f'  TP  {tp:6d}   paper {tp_ref:6d}')
    print(f'  FP  {fp:6d}   paper {fp_ref:6d}')
    print(f'  GT  {tp + fn:6d}   paper {gt_ref:6d}')
    print(f'  IoU {iou:6.1f} % paper {iou_ref:6.1f} %')

# Sanity check on the paper's own arithmetic, no assets needed:
for name, (_, tp_ref, fp_ref, gt_ref, iou_ref) in PAPER.items():
    fn_ref = gt_ref - tp_ref
    assert abs(100.0 * tp_ref / (tp_ref + fp_ref + fn_ref) - iou_ref) < 0.05, name
print('\npaper TP/FP/GT reproduce its stated IoUs')